# T1 — The fact and the moment

**Question:** Have Italian real wages really stayed flat against European peers, and when did the gap open?

**Goal:** prove the fact and identify the breaking point before investigating the causes.

**Deliverable:** the interactive Plotly chart `site/viz/t1_italy_alone_in_europe.html` (*Italy alone in Europe*) — embedded as Figure 1 in the published report. Italy's real-wage trajectory plotted against four European peers + the OECD aggregate, indexed 1995 = 100, with period-aware annotations (full / pre-2008 / post-2008).

**Countries:** Italy as focal; Germany, France, Spain, Sweden as peers; OECD-Total as the broad benchmark — period 1995-2024 (productivity series starts in 1995).

## 1. Setup — libraries

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titleweight"] = "bold"

DATA_DIR = Path("data/task_1")
DATA_DIR.resolve()

PosixPath('/Users/fil/Desktop/Data Visualization/Data-visualization-project/data/task_1')

## 2. Import dataset 1 — OECD Average Annual Wages (real)

**Source:** OECD Earnings Database — `DSD_EARNINGS@AV_AN_WAGE` (v1.0)
[Indicator page](https://www.oecd.org/en/data/indicators/average-annual-wages.html) · [Data Explorer](https://data-explorer.oecd.org/?tm=average+annual+wages)
License: [OECD Terms and Conditions](https://www.oecd.org/termsandconditions/)

**Filters applied in the interface**

| Dimension | Selected value |
|---|---|
| Reference area | Belgium, France, Germany, Greece, Italy, Netherlands, Portugal, Spain, Sweden, United Kingdom, OECD (total) |
| Unit of measure | US dollars, PPP converted |
| Time period | 1990 — 2024 |
| Pay period | Annual |
| Sex | Not applicable (aggregate) |
| Aggregation operation | Mean |
| Price base | Constant prices *(applied by default)* |
| Measure | Wages |

**Coverage:** 374 observations · 11 countries (10 national + OECD-Total) · 35 years · ~97% coverage.

**Why these choices**
- *Italy* as focal case; *Germany, France, Spain, UK* as primary comparators; *Belgium, Netherlands, Sweden* for Northern Europe; *Greece, Portugal* for post-2008 Southern-European dynamics; *OECD-Total* as broad international benchmark.
- *1990–2024* spans pre-Maastricht through the post-COVID / post-inflation period.
- *USD PPP-converted + constant prices* is the standard pairing for international real-wage comparisons: PPPs correct for cost-of-living differences, constant prices deflate for inflation.
- *Sex = Not applicable* gives a both-sexes aggregate; gender disaggregation can be reintroduced later if needed.

**CSV structure (key columns)**

| Column | Type | Description |
|---|---|---|
| `REF_AREA` | str | ISO country code (e.g. ITA, DEU) |
| `Reference area` | str | Country full name |
| `MEASURE` | str | `WG` = Wages |
| `UNIT_MEASURE` | str | `USD_PPP` |
| `PAY_PERIOD` | str | `A` = Annual |
| `PRICE_BASE` | str | `Q` = Constant prices |
| `TIME_PERIOD` | int | Year (1990–2024) |
| `OBS_VALUE` | float | Average annual wage (constant USD, PPP-converted) |
| `BASE_PER` | int | Base year for constant prices (2024) |
| `OBS_STATUS` | str | `A` = Normal value |

**Known limitations**
- *Germany:* 1990 missing (reunification structural break) — use 1991 as base for indexed comparisons and flag in captions.
- *Greece, Portugal:* five years missing each (series start later than 1990) — annotate shorter spans in any viz including them.
- *OECD-Total:* includes non-European economies (US, Japan, Korea, …) — treat as background benchmark, not a direct peer.
- *OECD revisions:* dataset is revised retrospectively; cite the download date alongside the indicator code for reproducibility.

**File:** `data/task_1/wages_real_oecd_1990_2024.csv` — UTF-8, comma-separated.

In [2]:
wages_raw = pd.read_csv(DATA_DIR / "wages_real_oecd_1990_2024.csv")

print(f"shape: {wages_raw.shape}")
print(f"years: {wages_raw['TIME_PERIOD'].min()}–{wages_raw['TIME_PERIOD'].max()}")
print(f"countries: {wages_raw['Reference area'].nunique()}")
wages_raw.head()

shape: (374, 30)
years: 1990–2024
countries: 11


,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,MEASURE,Measure,UNIT_MEASURE,Unit of measure,PAY_PERIOD,Pay period,PRICE_BASE,Price base,AGGREGATION_OPERATION,Aggregation operation,SEX,Sex,TIME_PERIOD,Time period,OBS_VALUE,Observation value,BASE_PER,Base period,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals
0,DATAFLOW,OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0),Average annual wages,I,BEL,Belgium,WG,Wages,USD_PPP,"US dollars, PPP converted",A,Annual,Q,Constant prices,MEAN,Mean,_Z,Not applicable,1990,NaN,58012.692,NaN,2024,NaN,A,Normal value,0,Units,0,Zero
1,DATAFLOW,OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0),Average annual wages,I,BEL,Belgium,WG,Wages,USD_PPP,"US dollars, PPP converted",A,Annual,Q,Constant prices,MEAN,Mean,_Z,Not applicable,1991,NaN,60561.112,NaN,2024,NaN,A,Normal value,0,Units,0,Zero
2,DATAFLOW,OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0),Average annual wages,I,BEL,Belgium,WG,Wages,USD_PPP,"US dollars, PPP converted",A,Annual,Q,Constant prices,MEAN,Mean,_Z,Not applicable,1992,NaN,62503.058,NaN,2024,NaN,A,Normal value,0,Units,0,Zero
3,DATAFLOW,OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0),Average annual wages,I,BEL,Belgium,WG,Wages,USD_PPP,"US dollars, PPP converted",A,Annual,Q,Constant prices,MEAN,Mean,_Z,Not applicable,1993,NaN,63816.775,NaN,2024,NaN,A,Normal value,0,Units,0,Zero
4,DATAFLOW,OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0),Average annual wages,I,BEL,Belgium,WG,Wages,USD_PPP,"US dollars, PPP converted",A,Annual,Q,Constant prices,MEAN,Mean,_Z,Not applicable,1994,NaN,65054.923,NaN,2024,NaN,A,Normal value,0,Units,0,Zero


## 3. Import dataset 2 — OECD GDP per Hour Worked (labour productivity)

**Source:** OECD Productivity Database — `DSD_PDB@DF_PDB_LV` (v1.0)
[Indicator page](https://www.oecd.org/en/data/indicators/gdp-per-hour-worked.html) · [Data Explorer](https://data-explorer.oecd.org/)
License: [OECD Terms and Conditions](https://www.oecd.org/termsandconditions/)

**Filters applied in the interface**

| Dimension | Selected value |
|---|---|
| Reference area | Belgium, France, Germany, Greece, Italy, Netherlands, Portugal, Spain, Sweden, United Kingdom, OECD (total) |
| Measure | GDP per hour worked |
| Unit of measure | US dollars per hour, PPP converted |
| Price base | Constant prices |
| Frequency | Annual |
| Economic activity | Total — all activities |
| Time period | 1990 — 2024 *(effective coverage in data: 1995–2024)* |
| Base period | 2020 |

**Coverage:** 325 observations · 11 countries (10 national + OECD-Total) · 30 years effectively (1995–2024) · ~97% coverage.

**Why these choices**
- *Same country panel as T1.1:* mismatched panels would invalidate the wage–productivity decoupling analysis.
- *GDP per hour worked* (not per person employed): isolates productivity from differences in average working hours across countries (Italy vs. Germany differ noticeably).
- *USD PPP-converted + constant prices:* standard pairing for international real-productivity comparisons.
- *Activity = Total:* whole-economy view; sectoral disaggregation is reserved for T3.

**CSV structure (key columns)**

| Column | Type | Description |
|---|---|---|
| `REF_AREA` | str | ISO country code |
| `Reference area` | str | Country full name |
| `MEASURE` | str | `GDPHRS` = GDP per hour worked |
| `UNIT_MEASURE` | str | `USD_PPP_H` |
| `PRICE_BASE` | str | `Q` = Constant prices |
| `FREQ` | str | `A` = Annual |
| `ACTIVITY` | str | `_T` = Total all activities |
| `TIME_PERIOD` | int | Year (1995–2024) |
| `OBS_VALUE` | float | GDP per hour worked (constant USD, PPP-converted) |
| `BASE_PER` | int | Base year for constant prices (2020) |
| `OBS_STATUS` | str | `A` = Normal value |

**Known limitations**
- *Coverage starts in 1995, not 1990:* OECD doesn't provide harmonised hourly productivity for European countries before 1995. When paired with the wage series (T1.1, from 1990), the joint comparison must use **1995 as the common base year** — this truncates the first five years of the wage analysis.
- *Different base year vs T1.1:* productivity is in 2020 constant USD, wages in 2024 constant USD. Doesn't affect indexed comparisons (1995 = 100) but must be flagged when showing absolute values side-by-side.
- *OECD-Total has six missing years* (likely 1995–2000) — treat as illustrative, not strict comparator.
- *OECD revisions:* productivity series is revised retrospectively (e.g. on GDP reweightings) — cite download date alongside the indicator code for reproducibility.

**File:** `data/task_1/productivity_gdp_per_hour_oecd_1995_2024.csv` — UTF-8, comma-separated.

In [3]:
prod_raw = pd.read_csv(DATA_DIR / "productivity_gdp_per_hour_oecd_1995_2024.csv")

print(f"shape: {prod_raw.shape}")
print(f"years: {prod_raw['TIME_PERIOD'].min()}–{prod_raw['TIME_PERIOD'].max()}")
print(f"countries: {prod_raw['Reference area'].nunique()}")
prod_raw.head()

shape: (324, 34)
years: 1995–2024
countries: 11


,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,FREQ,Frequency of observation,MEASURE,Measure,ACTIVITY,Economic activity,UNIT_MEASURE,Unit of measure,PRICE_BASE,Price base,TRANSFORMATION,Transformation,ADJUSTMENT,Adjustment,CONVERSION_TYPE,Conversion type,TIME_PERIOD,Time period,OBS_VALUE,Observation value,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,BASE_PER,Base period,DECIMALS,Decimals
0,DATAFLOW,OECD.SDD.TPS:DSD_PDB@DF_PDB_LV(1.0),Productivity levels,I,BEL,Belgium,A,Annual,GDPHRS,GDP per hour worked,_T,Total - all activities,USD_PPP_H,"US dollars per hour, PPP converted",Q,Constant prices,_Z,Not applicable,_Z,Not applicable,_Z,Not applicable,1995,NaN,72.001964,NaN,A,Normal value,0,Units,2020,NaN,2,Two
1,DATAFLOW,OECD.SDD.TPS:DSD_PDB@DF_PDB_LV(1.0),Productivity levels,I,BEL,Belgium,A,Annual,GDPHRS,GDP per hour worked,_T,Total - all activities,USD_PPP_H,"US dollars per hour, PPP converted",Q,Constant prices,_Z,Not applicable,_Z,Not applicable,_Z,Not applicable,1996,NaN,72.935803,NaN,A,Normal value,0,Units,2020,NaN,2,Two
2,DATAFLOW,OECD.SDD.TPS:DSD_PDB@DF_PDB_LV(1.0),Productivity levels,I,BEL,Belgium,A,Annual,GDPHRS,GDP per hour worked,_T,Total - all activities,USD_PPP_H,"US dollars per hour, PPP converted",Q,Constant prices,_Z,Not applicable,_Z,Not applicable,_Z,Not applicable,1997,NaN,74.845333,NaN,A,Normal value,0,Units,2020,NaN,2,Two
3,DATAFLOW,OECD.SDD.TPS:DSD_PDB@DF_PDB_LV(1.0),Productivity levels,I,BEL,Belgium,A,Annual,GDPHRS,GDP per hour worked,_T,Total - all activities,USD_PPP_H,"US dollars per hour, PPP converted",Q,Constant prices,_Z,Not applicable,_Z,Not applicable,_Z,Not applicable,1998,NaN,75.361964,NaN,A,Normal value,0,Units,2020,NaN,2,Two
4,DATAFLOW,OECD.SDD.TPS:DSD_PDB@DF_PDB_LV(1.0),Productivity levels,I,BEL,Belgium,A,Annual,GDPHRS,GDP per hour worked,_T,Total - all activities,USD_PPP_H,"US dollars per hour, PPP converted",Q,Constant prices,_Z,Not applicable,_Z,Not applicable,_Z,Not applicable,1999,NaN,76.912571,NaN,A,Normal value,0,Units,2020,NaN,2,Two


## 4. Build long-format analytical panels

The OECD raw downloads come as wide tables with many descriptive columns we don't need. We extract a clean long format (`iso3`, `country`, `year`, `value`) for each indicator and standardise column names. This is the foundation for the joint wage–productivity panel below.

In [4]:
# Long format — wages
wages_long = (
    wages_raw[["REF_AREA", "Reference area", "TIME_PERIOD", "OBS_VALUE"]]
    .rename(columns={
        "REF_AREA": "iso3",
        "Reference area": "country",
        "TIME_PERIOD": "year",
        "OBS_VALUE": "wage_real_usd_ppp",
    })
    .dropna(subset=["wage_real_usd_ppp"])
    .reset_index(drop=True)
)

# Long format — productivity
prod_long = (
    prod_raw[["REF_AREA", "Reference area", "TIME_PERIOD", "OBS_VALUE"]]
    .rename(columns={
        "REF_AREA": "iso3",
        "Reference area": "country",
        "TIME_PERIOD": "year",
        "OBS_VALUE": "prod_per_hour_usd_ppp",
    })
    .dropna(subset=["prod_per_hour_usd_ppp"])
    .reset_index(drop=True)
)

print(f"wages_long: {wages_long.shape}")
print(f"prod_long:  {prod_long.shape}")
wages_long.head()

wages_long: (374, 4)
prod_long:  (324, 4)


,iso3,country,year,wage_real_usd_ppp
0,BEL,Belgium,1990,58012.692
1,BEL,Belgium,1991,60561.112
2,BEL,Belgium,1992,62503.058
3,BEL,Belgium,1993,63816.775
4,BEL,Belgium,1994,65054.923


## 5. Build the joint wage–productivity panel

We outer-join wages and productivity on `(iso3, country, year)` and restrict to 1995+ (the productivity series starts in 1995). Each country is then indexed to its own 1995 = 100 value for both series, so the cross-country comparison is invariant to absolute wage / productivity levels.

In [5]:
# Outer-join wages and productivity, restrict to 1995+ for indexed analysis
panel_t1_1995 = (
    wages_long.merge(prod_long, on=["iso3", "country", "year"], how="outer")
              .query("year >= 1995")
              .reset_index(drop=True)
)

# Index each country to 1995 = 100 for both series
def index_to_base(df, value_col, base_year=1995, key_col="country"):
    """Add a *_idx column where each country's series is normalised to base_year=100."""
    bases = (df[df["year"] == base_year]
             .drop_duplicates(key_col)
             .set_index(key_col)[value_col])
    out = df.copy()
    out[f"{value_col}_idx"] = out.apply(
        lambda r: 100 * r[value_col] / bases.get(r[key_col])
                  if pd.notna(r[value_col]) and r[key_col] in bases.index
                  else np.nan,
        axis=1,
    )
    return out

panel_t1_1995 = index_to_base(panel_t1_1995, "wage_real_usd_ppp")
panel_t1_1995 = index_to_base(panel_t1_1995, "prod_per_hour_usd_ppp")

print(f"panel_t1_1995 shape: {panel_t1_1995.shape}")
panel_t1_1995.head()

panel_t1_1995 shape: (330, 7)


,iso3,country,year,wage_real_usd_ppp,prod_per_hour_usd_ppp,wage_real_usd_ppp_idx,prod_per_hour_usd_ppp_idx
0,BEL,Belgium,1995,64833.577,72.001964,100.000000,100.000000
1,BEL,Belgium,1996,65646.363,72.935803,101.253650,101.296963
2,BEL,Belgium,1997,66483.432,74.845333,102.544754,103.949016
3,BEL,Belgium,1998,66523.307,75.361964,102.606258,104.666540
4,BEL,Belgium,1999,70231.232,76.912571,108.325401,106.820101


## 6. Sanity check — does the headline story hold?

Before drawing any chart we verify the empirical claim at the heart of the research question. The check has three parts:

1. **Italy 1995→2024:** real wages should be flat or slightly positive; productivity should be flat or slightly positive (both growing very slowly, by international standards).
2. **Comparators (DE / FR / ES / UK / SE):** wages should grow significantly, productivity should grow even more.
3. **Italy as outlier:** Italy should rank last in both wage growth and productivity growth among the European peers.

If any of these fails, the research framing must be revisited before producing visualisations.

In [6]:
focal_countries = ["Italy", "Germany", "France", "Spain", "Sweden"]

# Compute 1995 -> 2024 growth in indexed terms (idx_2024 - 100)
def growth_table(panel):
    rows = []
    for c in focal_countries:
        sub = panel[panel["country"] == c]
        if sub.empty:
            continue
        v_2024 = sub[sub["year"] == 2024]["wage_real_usd_ppp_idx"].squeeze()
        p_2024 = sub[sub["year"] == 2024]["prod_per_hour_usd_ppp_idx"].squeeze()
        rows.append({
            "country": c,
            "wage_growth_pct": v_2024 - 100 if pd.notna(v_2024) else np.nan,
            "prod_growth_pct": p_2024 - 100 if pd.notna(p_2024) else np.nan,
        })
    return pd.DataFrame(rows)

growth = growth_table(panel_t1_1995).sort_values("wage_growth_pct")
print("Real-wage and productivity growth, 1995 -> 2024 (% change, indexed)")
print("=" * 72)
print(growth.round(1).to_string(index=False))
print()
print("Headline finding: Italy ranks last on both metrics among the five peers.")

Real-wage and productivity growth, 1995 -> 2024 (% change, indexed)
country  wage_growth_pct  prod_growth_pct
  Italy              2.7              6.5
  Spain              4.8             19.4
Germany             21.5             33.6
 France             26.6             26.0
 Sweden             62.1             47.2

Headline finding: Italy ranks last on both metrics among the five peers.


## 7. Save the consolidated T1 panel

The merged, cleaned and indexed panel is saved to `data/task_1/panel_t1_wages_productivity_1995_2024.csv` as the analytical archive for the task. Schema:

| Column | Description |
|---|---|
| `iso3` | OECD ISO3 country code |
| `country` | Country full name |
| `year` | Calendar year (1995–2024) |
| `wage_real_usd_ppp` | Real average annual wage, constant USD PPP (OECD AWE) |
| `prod_per_hour_usd_ppp` | Labour productivity, GDP per hour worked, constant USD PPP (OECD PDB) |
| `wage_real_usd_ppp_idx` | Wage indexed to 1995 = 100 (per country) |
| `prod_per_hour_usd_ppp_idx` | Productivity indexed to 1995 = 100 (per country) |

In [7]:
OUT_PATH = Path("data/task_1") / "panel_t1_wages_productivity_1995_2024.csv"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

panel_t1_1995.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")
print(f"Shape: {panel_t1_1995.shape}")
print(f"Columns: {panel_t1_1995.columns.tolist()}")

Saved: data/task_1/panel_t1_wages_productivity_1995_2024.csv
Shape: (330, 7)
Columns: ['iso3', 'country', 'year', 'wage_real_usd_ppp', 'prod_per_hour_usd_ppp', 'wage_real_usd_ppp_idx', 'prod_per_hour_usd_ppp_idx']


## 8. Final interactive visualisation — *Italy: alone in Europe*

Deliverable interactive chart embedded in the newspaper article.

**Design decisions** (locked in `02_viz_plan.md`, T1, after iterative review):

- **Single line chart**, indexed 1995 = 100 — the project's only allowed line chart.
- **Distinct colour per country.** Italy in saturated editorial red; peers in muted hues (slate blue, terracotta, sage, plum); OECD as dashed dark grey.
- **Italy emphasis: thicker line + white outline.** Italy is rendered at 4.5 px with a 6.5 px paper-colour outline beneath it, which acts as a clean visual separator from the peer lines without resembling an error band.
- **Endpoint markers with period-aware values.** Each line ends with a coloured dot and the cumulative `% change` for the active period, floating just past the marker. The values update with the preset buttons.
- **Legend in the top-right corner**, where the lines reach their final heights — less visual collision with the early years where lines are densely packed near the 1995 baseline.
- **Translucent red gap band** between Italy and the peer median visualises *what Italy missed*.
- **Three macro shocks annotated**: 2008 GFC, 2011 sovereign-debt crisis, 2020 COVID-19 (with CIG note).
- **Three preset buttons** — *Full period*, *Pre-2008*, *Post-2008* — swap axis ranges, endpoint positions and values, and legend percentages atomically.

**Tool:** Plotly (Python). Output: `site/viz/t1_italy_alone_in_europe.html`.

### 8.1 Visual identity and country panel

In [8]:
import plotly.graph_objects as go
import plotly.io as pio

# Visual identity
COLOR_ITALY     = "#c44536"
COLOR_GERMANY   = "#4a6fa5"
COLOR_FRANCE    = "#c97c5d"
COLOR_SPAIN     = "#6b8e7f"
COLOR_SWEDEN    = "#8d6b94"
COLOR_OECD      = "#555555"
COLOR_OUTLINE   = "#f8f5ec"   # same as paper background — used as Italy outline
COLOR_GAP_FILL  = "rgba(196, 69, 54, 0.10)"
COLOR_GRID      = "#e8e8e8"
COLOR_RULE      = "#cccccc"
COLOR_BG_PAPER  = "#f8f5ec"
COLOR_TOOLTIP   = "#1a3a5c"

# Trace order matters — Italy is rendered LAST (on top of peers)
COUNTRY_ORDER = [
    ("Germany", COLOR_GERMANY,  1.8, "solid"),
    ("France",  COLOR_FRANCE,   1.8, "solid"),
    ("Spain",   COLOR_SPAIN,    1.8, "solid"),
    ("Sweden",  COLOR_SWEDEN,   1.8, "solid"),
    ("OECD",    COLOR_OECD,     1.6, "dash"),
    ("Italy",   COLOR_ITALY,    4.5, "solid"),  # focal: thick, on top
]
COUNTRY_COLORS = {n: c for n, c, *_ in COUNTRY_ORDER}

# Display order in the legend — Italy first, then by 2024 ranking
LEGEND_ORDER = ["Italy", "Sweden", "OECD", "France", "Germany", "Spain"]

SHOCK_EVENTS = [
    (2008, "2008<br>Global financial<br>crisis"),
    (2011, "2011<br>Sovereign-debt<br>crisis"),
    (2020, "2020<br>COVID-19"),
]

PEER_FOR_BAND = ["Germany", "France", "Spain", "Sweden"]

### 8.2 Compute peer median and period-aware metrics

For each country we cache, per period, both the **endpoint year**, the **endpoint indexed value** (the y position) and the **percentage change** (what the endpoint label displays). The buttons later use these to reposition markers and update labels.

In [9]:
# Peer median for the gap band
peer_pivot = (
    panel_t1_1995[panel_t1_1995["country"].isin(PEER_FOR_BAND)]
    .pivot_table(index="year", columns="country", values="wage_real_usd_ppp_idx")
)
peer_median = peer_pivot.median(axis=1)

italy_series = (
    panel_t1_1995[panel_t1_1995["country"] == "Italy"]
    .set_index("year")["wage_real_usd_ppp_idx"]
)


def country_year_value(country: str, year: int):
    sub = panel_t1_1995[panel_t1_1995["country"] == country]
    s = sub[sub["year"] == year]
    if s.empty: return None
    v = s.iloc[0]["wage_real_usd_ppp_idx"]
    return None if pd.isna(v) else float(v)

def pct_change(country: str, start_year: int, end_year: int):
    a = country_year_value(country, start_year)
    b = country_year_value(country, end_year)
    if a is None or b is None or a == 0: return None
    return (b / a - 1) * 100


# For each preset: (end_year, start_year_for_pct)
PRESETS = {
    "full": dict(end_year=2024, start_year=1995),
    "pre":  dict(end_year=2008, start_year=1995),
    "post": dict(end_year=2024, start_year=2008),
}

# Cache per-country endpoint info per preset:
#   endpoints[preset][country] = {"x": end_year, "y": idx_value, "pct": cumulative %}
endpoints = {}
for preset, cfg in PRESETS.items():
    endpoints[preset] = {}
    for country in LEGEND_ORDER:
        y = country_year_value(country, cfg["end_year"])
        p = pct_change(country, cfg["start_year"], cfg["end_year"])
        endpoints[preset][country] = {
            "x": cfg["end_year"],
            "y": y,
            "pct": p,
        }

print("Full  →", {c: f"y={d['y']:.1f} ({d['pct']:+.0f}%)" for c, d in endpoints["full"].items()})
print("Pre   →", {c: f"y={d['y']:.1f} ({d['pct']:+.0f}%)" for c, d in endpoints["pre"].items()})
print("Post  →", {c: f"y={d['y']:.1f} ({d['pct']:+.0f}%)" for c, d in endpoints["post"].items()})

Full  → {'Italy': 'y=102.7 (+3%)', 'Sweden': 'y=162.1 (+62%)', 'OECD': 'y=130.6 (+31%)', 'France': 'y=126.6 (+27%)', 'Germany': 'y=121.5 (+22%)', 'Spain': 'y=104.8 (+5%)'}
Pre   → {'Italy': 'y=108.8 (+9%)', 'Sweden': 'y=142.1 (+42%)', 'OECD': 'y=116.6 (+17%)', 'France': 'y=114.8 (+15%)', 'Germany': 'y=106.1 (+6%)', 'Spain': 'y=102.4 (+2%)'}
Post  → {'Italy': 'y=102.7 (-6%)', 'Sweden': 'y=162.1 (+14%)', 'OECD': 'y=130.6 (+12%)', 'France': 'y=126.6 (+10%)', 'Germany': 'y=121.5 (+15%)', 'Spain': 'y=104.8 (+2%)'}


### 8.3 Build the figure — gap band, peers, Italy outline + main

In [10]:
fig = go.Figure()

# ---- 1. Gap band (closed polygon) ----
band_x = list(peer_median.index) + list(italy_series.index[::-1])
band_y = list(peer_median.values) + list(italy_series.values[::-1])
fig.add_trace(go.Scatter(
    x=band_x, y=band_y,
    fill="toself", fillcolor=COLOR_GAP_FILL,
    line=dict(width=0),
    showlegend=False, hoverinfo="skip", name="gap_band",
))

# ---- 2. Country lines (Italy goes last) ----
italy_data = (
    panel_t1_1995[panel_t1_1995["country"] == "Italy"]
    .dropna(subset=["wage_real_usd_ppp_idx"])
    .sort_values("year")
)

for country, color, width, dash in COUNTRY_ORDER:
    sub = (
        panel_t1_1995[panel_t1_1995["country"] == country]
        .dropna(subset=["wage_real_usd_ppp_idx"])
        .sort_values("year")
    )

    hover = []
    for _, r in sub.iterrows():
        chg = r["wage_real_usd_ppp_idx"] - 100
        sgn = "+" if chg >= 0 else "−"
        hover.append(
            f"<b>{country}</b><br>"
            f"Year: {int(r['year'])}<br>"
            f"Index (1995=100): {r['wage_real_usd_ppp_idx']:.1f}<br>"
            f"Cumulative change vs 1995: {sgn}{abs(chg):.1f}%"
        )

    # ---- White outline rendered just before the Italy main line ----
    if country == "Italy":
        fig.add_trace(go.Scatter(
            x=sub["year"], y=sub["wage_real_usd_ppp_idx"],
            mode="lines",
            line=dict(color=COLOR_OUTLINE, width=width + 2),
            showlegend=False, hoverinfo="skip", name="italy_outline",
        ))

    fig.add_trace(go.Scatter(
        x=sub["year"], y=sub["wage_real_usd_ppp_idx"],
        name=country, mode="lines",
        line=dict(color=color, width=width, dash=dash),
        hovertext=hover, hoverinfo="text",
        showlegend=False,
    ))

print(f"Figure built — {len(fig.data)} traces.")

Figure built — 8 traces.


### 8.4 Custom legend (top-right) and endpoint annotations

The legend (top-right) lists countries with their colour swatch — no percentages. The percentages live at the **endpoint annotations**: a coloured dot followed by the period-aware value (e.g. *● +62 %* for Sweden). Both the legend and the endpoints are rebuilt as full annotation arrays for each preset, swapped atomically by the buttons.

In [11]:
def legend_html(countries):
    """Build the legend HTML — dashed glyph for OECD, solid for the rest."""
    rows = []
    for c in countries:
        col = COUNTRY_COLORS[c]
        glyph = "╌╌╌" if c == "OECD" else "━━"   # dashed glyph for the OECD benchmark
        rows.append(
            f"<span style=\"color:{col};font-weight:900;letter-spacing:-1px;\">{glyph}</span>"
            f"&nbsp;&nbsp;<span style=\"color:#222;\">{c}</span>"
        )
    return "<br>".join(rows)


def build_annotations(preset_key: str) -> list:
    anns = []
    pe = endpoints[preset_key]

    # Shock annotations
    for yr, label in SHOCK_EVENTS:
        anns.append(dict(
            x=yr, y=170, xref="x", yref="y",
            text=label, showarrow=False,
            font=dict(family="EB Garamond, serif", size=10, color="#888"),
            align="center", yanchor="top",
        ))

    # 1995 baseline label (positioned just above the chart on the right)
    anns.append(dict(
        x=2025.4, y=100, xref="x", yref="y",
        text="1995 base", showarrow=False,
        font=dict(family="EB Garamond, serif", size=10, color="#999"),
        xanchor="right", yanchor="middle",
    ))

    # Source caption
    anns.append(dict(
        x=0.02, y=-0.22, xref="paper", yref="paper",
        text=("Source: OECD Average Annual Wages (constant USD PPP). "
              "Hover any line for country-year values."),
        showarrow=False,
        font=dict(family="EB Garamond, serif", size=11,
                  color="#666", style="italic"),
        xanchor="left",
    ))

    # ---- Legend (top-right corner, country list only — no %) ----
    anns.append(dict(
        x=0.025, y=0.94, xref="paper", yref="paper",
        text=legend_html(LEGEND_ORDER),
        showarrow=False,
        align="left",
        xanchor="left", yanchor="top",
        font=dict(family="Inter, sans-serif", size=11),
        bgcolor="rgba(250,250,250,0.92)",
        bordercolor="#cccccc",
        borderwidth=1,
        borderpad=8,
    ))

    # ---- Endpoint annotations (one per country) ----
    # Manual y-shifts for tightly stacked endpoints to avoid label collisions
    yshifts = {"full":  {"Italy": -7, "Spain": +8, "Germany": -6, "France": 0, "OECD": +8},
               "pre":   {"Italy": -2, "Spain": +6, "Germany": +1, "France": +1, "OECD": +6},
               "post":  {"Italy": -7, "Spain": +8, "Germany": -6, "France": 0, "OECD": +8}}

    for country in LEGEND_ORDER:
        d = pe[country]
        if d["y"] is None or d["pct"] is None: continue
        col = COUNTRY_COLORS[country]
        sgn = "+" if d["pct"] >= 0 else "−"
        text = (
            f"<span style=\"color:{col};font-size:14px;font-weight:900;\">●</span>"
            f"&nbsp;<b>{sgn}{abs(d['pct']):.0f}%</b>"
        )
        anns.append(dict(
            x=d["x"] + 0.4, y=d["y"], xref="x", yref="y",
            text=text, showarrow=False,
            font=dict(family="Inter, sans-serif", size=11, color=col),
            xanchor="left", yanchor="middle",
            yshift=yshifts.get(preset_key, {}).get(country, 0),
        ))

    return anns


fig.update_layout(
    title=dict(
        text=(
            "<b>Italy: alone in Europe</b><br>"
            "<span style='font-family:EB Garamond,serif;font-style:italic;"
            "font-size:14px;color:#666;'>"
            "Real wages, indexed to 1995 = 100. "
            "The shaded band shows the gap between Italy and the peer median."
            "</span>"
        ),
        x=0.02, y=0.95, xanchor="left",
        font=dict(family="Playfair Display, serif", size=22, color="#1a1a1a"),
    ),
    xaxis=dict(
        range=[1994.5, 2026.5],
        showgrid=False, zeroline=False,
        tickfont=dict(family="Inter, sans-serif", size=11, color="#444"),
    ),
    yaxis=dict(
        range=[94, 175],
        showgrid=True, gridcolor=COLOR_GRID, gridwidth=1, zeroline=False,
        tickfont=dict(family="Inter, sans-serif", size=11, color="#444"),
        title=dict(text="Real wages, index 1995 = 100",
                   font=dict(family="Inter, sans-serif", size=11, color="#444")),
    ),
    plot_bgcolor=COLOR_BG_PAPER,
    paper_bgcolor=COLOR_BG_PAPER,
    hoverlabel=dict(
        bgcolor=COLOR_TOOLTIP, bordercolor=COLOR_TOOLTIP,
        font=dict(family="Inter, sans-serif", size=12, color="white"),
    ),
    margin=dict(l=80, r=80, t=110, b=100),
    height=620,
    showlegend=False,
    annotations=build_annotations("full"),
    updatemenus=[
        dict(
            type="buttons", direction="right",
            x=0.02, y=-0.05, xanchor="left", yanchor="top",
            showactive=True,
            font=dict(family="Inter, sans-serif", size=11, color="#444"),
            bgcolor="#f0f0f0", bordercolor="#cccccc", borderwidth=1,
            pad=dict(r=8, t=4, b=4, l=8),
            buttons=[
                dict(label="Full period", method="relayout",
                     args=[{"xaxis.range": [1994.5, 2026.5],
                            "yaxis.range": [94, 175],
                            "annotations": build_annotations("full")}]),
                dict(label="Pre-2008", method="relayout",
                     args=[{"xaxis.range": [1994.5, 2010.5],
                            "yaxis.range": [94, 145],
                            "annotations": build_annotations("pre")}]),
                dict(label="Post-2008", method="relayout",
                     args=[{"xaxis.range": [2007.5, 2026.5],
                            "yaxis.range": [94, 175],
                            "annotations": build_annotations("post")}]),
            ],
        )
    ],
)

fig.show()

### 8.5 Export to standalone HTML

In [12]:
import json as _json

VIZ_OUT = Path("site/viz/t1_italy_alone_in_europe.html")
VIZ_OUT.parent.mkdir(parents=True, exist_ok=True)

pio.write_html(
    fig, str(VIZ_OUT),
    include_plotlyjs="cdn",
    full_html=True,
    config={"displayModeBar": False, "responsive": True},
)

# ---- T1 → T5 cross-chart linking (postMessage) ---------------------
# Append a tiny script that listens for clicks on country lines and posts
# `{event: 't5:set_country', iso3: <ISO>}` to the parent frame, which then
# forwards it to T5's iframe. T5 already has the receiver coded.
COUNTRY_TO_ISO = {
    "Italy": "ITA",
    "Germany": "DEU",
    "France": "FRA",
    "Spain": "ESP",
    "United Kingdom": "GBR",
    "Sweden": "SWE",
    "Belgium": "BEL",
    "Netherlands": "NLD",
    "Greece": "GRC",
    "Portugal": "PRT",
    "OECD": "OECD",
}
_iso_json = _json.dumps(COUNTRY_TO_ISO)

postmsg_script = (
    "<script>\n"
    "/* T1 → T5 cross-chart linking. When a country line is clicked, post a\n"
    "   message to the parent so T5's calculator iframe can pre-select that\n"
    "   country as the comparator. T5's receiver lives in t5_calculator.html. */\n"
    "(function () {\n"
    "  function attach() {\n"
    "    var gd = document.querySelector('.js-plotly-plot, .plotly-graph-div');\n"
    "    if (!gd || typeof gd.on !== 'function') return setTimeout(attach, 80);\n"
    "    var COUNTRY_TO_ISO = " + _iso_json + ";\n"
    "    gd.style.cursor = 'pointer';\n"
    "    gd.on('plotly_click', function (data) {\n"
    "      if (!data || !data.points || !data.points.length) return;\n"
    "      var country = data.points[0].data.name;\n"
    "      var iso = COUNTRY_TO_ISO[country];\n"
    "      // Skip Italy (the baseline) and OECD (aggregate, not a single country)\n"
    "      if (!iso || iso === 'ITA' || iso === 'OECD') return;\n"
    "      if (window.parent && window.parent !== window) {\n"
    "        window.parent.postMessage({ event: 't5:set_country', iso3: iso }, '*');\n"
    "      }\n"
    "    });\n"
    "  }\n"
    "  attach();\n"
    "})();\n"
    "</script>\n"
)

_html = VIZ_OUT.read_text()
_html = _html.replace("</body>", postmsg_script + "</body>")
VIZ_OUT.write_text(_html)

print(f"Saved: {VIZ_OUT}")
print(f"Size: {VIZ_OUT.stat().st_size / 1024:.1f} KB")
print("→ postMessage hook injected for T1 → T5 cross-chart linking.")

# --- centre the chart div within its iframe (post-processing) ---
_centering_css = (
    "<style>\n"
    "  html, body { margin: 0; padding: 0; }\n"
    "  .plotly-graph-div { margin: 0 auto !important; }\n"
    "</style>\n"
)
_html = open(str(VIZ_OUT)).read()
if "plotly-graph-div { margin: 0 auto" not in _html:
    _html = _html.replace("</head>", "</head>\n" + _centering_css)
    open(str(VIZ_OUT), "w").write(_html)



Saved: site/viz/t1_italy_alone_in_europe.html
Size: 59.9 KB
→ postMessage hook injected for T1 → T5 cross-chart linking.
